# RAG Study Assistant — Browser Chat (Mobile Friendly)

Run this notebook on **Google Colab** to chat with your study files from any phone or tablet.

**What you need**
1. A free [Google AI Studio API key](https://aistudio.google.com/apikey) (Gemini)
2. Study files in `files/` (copied to Google Drive on first run — see below)

**Google Drive (recommended)** — saves your vector index and embedding model cache so you **don't re-index every session**. First run takes a few minutes; later runs load in seconds.

**Steps:** Run each cell in order. After the last cell, open the `gradio.live` public link on your phone.

In [ ]:
# Install dependencies (pins OpenTelemetry for Colab's pre-installed google-adk)
%pip install -q --upgrade-strategy only-if-needed \
  "gradio>=4.44,<6" \
  "chromadb>=0.5,<0.6" \
  "posthog<4" \
  "sentence-transformers>=3,<4" \
  "google-generativeai>=0.8" \
  "opentelemetry-api>=1.36.0,<1.39.0" \
  "opentelemetry-sdk>=1.36.0,<1.39.0" \
  "opentelemetry-exporter-otlp-proto-common==1.38.0" \
  "opentelemetry-proto==1.38.0" \
  "opentelemetry-exporter-otlp-proto-http==1.38.0"

# Quick sanity check — if this prints OK, you can ignore any leftover pip noise above
import gradio
import chromadb
import google.generativeai as genai
print("OK — gradio", gradio.__version__, "| chromadb", chromadb.__version__)

In [ ]:
# Get the project code
# Option A — clone from GitHub (replace with your repo URL):
# !git clone https://github.com/YOUR_USER/codex_rag_runner.git
# %cd codex_rag_runner

# Option B — upload a zip of this repo, then uncomment:
# from google.colab import files
# uploaded = files.upload()  # upload codex_rag_runner.zip
# !unzip -q codex_rag_runner.zip
# %cd codex_rag_runner

import os
from pathlib import Path

if not Path('rag_runner').is_dir():
    raise SystemExit(
        'rag_runner package not found. Clone the repo or upload a zip, then %cd into the project folder.'
    )
print('Project root:', Path('.').resolve())

In [ ]:
# Mount Google Drive — persist index + model cache between Colab sessions
import shutil
from pathlib import Path

USE_DRIVE = True
DRIVE_ROOT = Path('/content/drive/MyDrive/rag_study')

RAG_PATHS = {
    'source_dir': './files',
    'index_dir': './index',
    'model_cache_dir': './model-cache',
}

if USE_DRIVE:
    from google.colab import drive

    drive.mount('/content/drive')

    for name in ('files', 'index', 'model-cache'):
        (DRIVE_ROOT / name).mkdir(parents=True, exist_ok=True)

    RAG_PATHS = {
        'source_dir': str(DRIVE_ROOT / 'files'),
        'index_dir': str(DRIVE_ROOT / 'index'),
        'model_cache_dir': str(DRIVE_ROOT / 'model-cache'),
    }

    # Copy bundled repo files to Drive on first run
    repo_files = Path('files')
    drive_files = Path(RAG_PATHS['source_dir'])
    if repo_files.is_dir():
        copied = []
        for src in sorted(repo_files.iterdir()):
            if src.is_file():
                dest = drive_files / src.name
                if not dest.exists():
                    shutil.copy2(src, dest)
                    copied.append(src.name)
        if copied:
            print('Copied to Drive files/:', ', '.join(copied))
        elif any(drive_files.iterdir()):
            print('Drive files/ already has study material.')
        else:
            print('Drive files/ is empty — upload study files in the next cell.')

    index_exists = any(Path(RAG_PATHS['index_dir']).rglob('*'))
    print('Drive root:', DRIVE_ROOT)
    print('Saved index on Drive:', 'yes (will skip re-indexing)' if index_exists else 'no (first run will build it)')
else:
    print('Using local paths only (index is lost when the Colab runtime ends).')

In [ ]:
# Set your Gemini API key (free tier works for study Q&A)
import os

try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    print('Loaded GEMINI_API_KEY from Colab secrets.')
except Exception:
    # Paste your key here if not using Colab secrets:
    # os.environ['GEMINI_API_KEY'] = 'YOUR_KEY_HERE'
    if not os.environ.get('GEMINI_API_KEY'):
        raise SystemExit(
            'Set GEMINI_API_KEY: Colab → 🔑 Secrets → add GEMINI_API_KEY, or paste in this cell.'
        )

In [ ]:
# Optional: upload extra study files (saved to Drive when USE_DRIVE=True)
# from google.colab import files
# uploaded = files.upload()
# from pathlib import Path
# target = Path(RAG_PATHS['source_dir'])
# target.mkdir(parents=True, exist_ok=True)
# for name, data in uploaded.items():
#     (target / name).write_bytes(data)
# print('Uploaded to', target, ':', list(uploaded.keys()))
# print('Click "Rebuild index" in the chat UI after adding new files.')

In [ ]:
# Launch chat UI with a public link for your phone
from rag_runner.chat_ui import launch_chat_ui

colab_config = {
    'llm': {
        'backend': 'gemini',
        'model': 'gemini-2.0-flash',
        'api_key_env': 'GEMINI_API_KEY',
    },
    'rag': {
        'source_dir': RAG_PATHS['source_dir'],
        'index_dir': RAG_PATHS['index_dir'],
        'model_cache_dir': RAG_PATHS['model_cache_dir'],
    },
    'runner': {'work_dir': './runs'},
}

launch_chat_ui(
    config=colab_config,
    share=True,
    host='0.0.0.0',
    port=7860,
)